In [ ]:
/*
=================================================================
      HANDLING TRAFFIC SPIKES - BEGINNER NOTES (HINGLISH)
=================================================================

-----------------------------------------------------------------
SECTION 1: TRAFFIC SPIKE KYA HAI? (Problem samjho pehle)
-----------------------------------------------------------------

PARAGRAPH:
Socho aapki dukaan pe normally roz 50 customers aate hain. Diwali
wale din achanak 500 aa gaye! Dukaan chhoti hai, ek salesman hai,
sab queue me lage, kuch customers gussa ho ke chale gaye. Yahi
traffic spike hai - system ke capacity se ZYADA demand ekdum se
aa jana. Example: aapki e-commerce site 1000 requests/second
handle karti hai, sale shuru hote hi 10,000 req/sec aa gaye.
Servers queue karne lagte hain, database bharta jata hai, aur
customers ko slow pages ya errors dikhte hain.

DIAGRAM:

    NORMAL DAY:
    1000 users ---> [ Server - capacity 1000 ] ---> Sab HAPPY :)

    SALE DAY (Spike!):
    10,000 users ---> [ Server - capacity 1000 ] ---> BHAGWAN BACHAYE
                          |
                          v
                    Requests QUEUE me
                    DB pool FULL
                    Users ko ERRORS

BULLET POINTS:
   * Spike = demand system ki capacity se TEZI se badh jana
   * HAR spike same nahi hota - news spike, sale spike, retry
     storm, DDoS attack - sabka ALAG ilaaj hai
   * Sirf "auto-scale + cache laga do" bolna kamzor answer hai
   * Strong answer = 4 cheezein:
        1. Scaling lag me survive kaise karoge
        2. Bottleneck (DB etc.) ko protect kaise karoge
        3. Extra kaam ko shed/queue kaise karoge
        4. Sabse important user flows (payment/checkout)
           kaise bachaoge

-----------------------------------------------------------------
SECTION 2: 3 TYPES OF SPIKES (Kaunsa spike, kaunsa ilaaj)
-----------------------------------------------------------------

PARAGRAPH:
Spikes 3 tarah ke hote hain. PREDICTABLE - aapko pata hai aa raha
hai (flash sale). Isme aap pehle se taiyari kar sakte ho. UNPREDICTABLE
- koi warning nahi (koi post viral ho gaya). Yahan sirf automatic
defenses kaam aati hain. SELF-INFLICTED - sabse dukh wala - aapka
APNA system hi spike banata hai (retry storms, cache expiry).
Jaise ghar ka meter hi short kar de!

DIAGRAM:

    [1] PREDICTABLE (Pata hai - taiyari kar sakte ho)
        Flash sale, product launch, TV finale, ad campaign
             |
             v
        PRE-SCALE + CACHE WARM karo

    [2] UNPREDICTABLE (No warning - auto defense chahiye)
        Viral post, breaking news, DDoS attack
             |
             v
        AUTO-DETECTION + AUTO-SCALING

    [3] SELF-INFLICTED (Aapka system hi culprit!)
        Cache expiry, service restart, retry storm
             |
             v
        FEEDBACK LOOP FIX karo (jitter, staggered TTL)

BULLET POINTS:
   * Predictable = proactive taiyari (scale, warm, rehearse)
   * Unpredictable = tez reaction chahiye, automatic systems
   * Self-inflicted = "thundering herd" - sab clients ek saath
     retry/cache-miss karte hain aur load X10 ho jata hai
   * Spike ka TYPE pehle classify karo, phir ilaaj socho!

-----------------------------------------------------------------
SECTION 3: SPIKE KI LIFE CYCle (Anatomy - 6 phases)
-----------------------------------------------------------------

PARAGRAPH:
Spike achanak aata hai aur achanak nahi jaata - uski ek life hoti
hai. RAMP-UP me traffic badh raha hai par naye servers abhi ready
nahi (DANGER ZONE). PEAK me maximum load, saare limits test ho
rahe hain. SUSTAINED sabse khatarnak - aap peak survive kar gaye,
par GHANTON tak high load chal raha hai - thread pools khatam,
memory leak dikhne lagti hai. RECOVERY bhi risky hai - queue ka
backlog, thande caches, DB ko catch-up time chahiye.

DIAGRAM:

    Load
     |                    PEAK (max load)
     |                   /|\
     |        RAMP-UP   / |  \  SUSTAINED (ghanton tak high)
     |       (danger!) /  |   \_______          DECLINE
     |      __________/   |        \  \_______________
     |_____/              |         \                 \____ RECOVERY
     |  NORMAL            |          |                      (risky!)
     +----------------------------------------------------------> Time
         naye servers     limits     threads/memory
         abhi ready       test ho    khatam ho raha
         NAHI hain        rahe hai   rahe hai

BULLET POINTS:
   * Ramp-up = naye servers launch ho rahe hain par ready nahi
   * Peak = headroom aur protection hai toh survive, nahi toh fail
   * Sustained = chhoti inefficiency bhi INCIDENT ban jati hai
   * Recovery = queues drain, caches rebuild, DB catch-up -
     jaldi me scale-down mat karna, apni problem bana sakte ho

-----------------------------------------------------------------
SECTION 4: SPIKE KHATARNAK KYUN HAI? (Cascade failure ka khel)
-----------------------------------------------------------------

PARAGRAPH:
System 10,000 req/sec handle karta hai. 15,000 pe sirf 50% slow
kyun nahi hota? Poori tarah FAIL kyun ho jata hai? Kyunki overload
me latency NON-LINEAR badhti hai. CPU 100% hua toh requests queue
hone lagti hain. Aur sabse bura: failures CASCADE karte hain - ek
component girta hai, baaki sabko kheench ke girata hai. Domino
effect! Aur math simple hai: 5,000 extra req/sec backlog = 1
minute me 3,00,000 queued requests. System kabhi catch-up nahi
kar sakta jab tak capacity/shedding na ho.

DIAGRAM (Vicious cycle - ek chakkar jise todna mushkil hai):

    +--> Database SLOW hua
    |         |
    |         v
    |    App ke threads DB ka wait karte rahi --> BLOCK
    |         |
    |         v
    |    Thread pool KHATAM (exhausted)
    |         |
    |         v
    |    Request queue FULL
    |         |
    |         v
    |    Load Balancer TIMEOUT deta hai
    |         |
    |         v
    |    Users RETRY karte hain  --> AUR ZYADA LOAD!
    |         |
    +---------+   (yeh loop chalta rehta hai = system-wide incident)

BULLET POINTS:
   * Har component ki ek LIMIT hai (CPU, memory, connections)
   * Limit cross hote hi latency LINEAR se NON-LINEAR ho jati hai
   * Ek component ka fail = poora chain girta hai (cascade)
   * Retry traffic SABSE KAMZOR component ko aur todta hai
   * Isliye "reject karna jaldi se" kabhi kabhi "30 sec hang hona"
     se BEHTAR hai

-----------------------------------------------------------------
SECTION 5: STRATEGY 1 - AUTO-SCALING (Automatic naukar bharti)
-----------------------------------------------------------------

PARAGRAPH:
Auto-scaling matlab - jab kaam badhe toh automatically naye servers
launch ho jayein, aur kaam kam ho toh extra servers band (cost
saving). Ye ek control loop hai: metrics dekho (CPU, memory,
latency, queue depth) -> threshold compare karo -> servers
add/remove karo. LEKIN sabse bada problem: SCALING LAG! Naye
server ko launch hone me 3-5 MINUTE lagte hain. Tab tak purane
servers hi poori load utha rahe hote hain. Agar wo pehle hi mar
chuke hain, toh naya server aane ka kya fayda?

DIAGRAM:

    CONTROL LOOP:
    [Metrics] --> CPU > 70%? --> [Auto-Scaler Controller]
    (CPU, memory,                      |
     latency, queue)          +-------+--------+
                              |                |
                         SCALE UP          SCALE DOWN
                         +Servers          -Servers

    SCALING LAG TIMELINE (yahi problem hai!):
    T+0 sec      Spike AA GAYA!      <-- abhi se system daba raha
    T+30 sec     Metrics collect hui
    T+60 sec     Scale decision liya
    T+180 sec    Instance launching...
    T+240 sec    App start ho raha...
    T+300 sec    Server READY!       <-- 5 MINUTE BAAD! Der ho gayi?

BULLET POINTS:
   * Policies: Target tracking (CPU 70%), Step scaling (zyada
     load = zyada servers), Scheduled (9AM baje capacity badho),
     Predictive (ML se future load ka andaza)
   * Spike ke liye STEP SCALING best hai
   * Pros: automatic, cost-efficient, gradual growth sambhalta hai
   * Cons: SLOW (minutes lagte hain), cold start latency,
     Sudden spike ke liye AKELA kaafi NAHI
   * Auto-scaling = defense ki ek LAYER hai, poori defense nahi!

-----------------------------------------------------------------
SECTION 6: STRATEGY 2 - LOAD SHEDDING (Kaun andar aayega, kaun nahi)
-----------------------------------------------------------------

PARAGRAPH:
System daba hua hai. Ab do raste hain: SABKO kharab service do
(sabka time waste), YA sabse important requests achhe se serve
karo aur baaki ko jhat se REJECT karo. Load shedding doosra rasta
chunta hai. Jaise restaurant full ho toh gate pe hi bolo "abhay
full hai, 30 min baad aana" - andar bitha ke khana NA dena behtar
hai! Fast rejection (503 + Retry-After header) client ko turant
pata chal jata hai. Slow timeout resources block karta rehta hai.
Aur PRIORITY based shedding - payment/analytics barabar nahi hote!

DIAGRAM:

    Incoming Requests
          |
          v
    +---------------+      Under capacity  --> ACCEPT (normal process)
    | LOAD SHEDDER  | --->                      (Server --> Database)
    | (gate keeper) |
    +---------------+      Over capacity   --> REJECT with:
          |                                     503 Service Unavailable
          v                                     Retry-After: 30 sec

    PRIORITY LADDER (load badhne pe neeche wale PEHLE bahar!):

    Load < 70%  : SAB accept
    Load 70-85% : P3 shed (analytics, metrics, batch jobs)
    Load 85-95% : P2 bhi shed (recommendations, reviews)
    Load > 95%  : SIRF P0 (payments, auth, checkout) hi bacha!

    P0: CRITICAL   - Payments, Auth, Checkout   (SABSE LAST shed)
    P1: Important  - Search, Product pages, Cart
    P2: Enhancement- Recommendations, Reviews, Personalization
    P3: Background - Analytics, Metrics, Batch  (SABSE PEHLE shed)

BULLET POINTS:
   * MANTRA: "controlled failure" > "sabki service kharab"
   * Edge pe (entry pe hi) reject karo - resource waste na ho
   * 503 + Retry-After return karo
   * Priorities CRISIS se PEHLE decide karo, crisis me nahi!
   * Shedding methods: Queue-depth based, Latency-based (P99 > 500ms),
     Adaptive (accept rate adjust karte raho)

    CLIENT SIDE KA CONTRACT (warna retry storm!):
    * Exponential BACKOFF + JITTER: retry 1s, 2s, 4s... aur thoda
      RANDOM gap (jitter) - warna SAB ek saath retry karenge aur
      NAYA storm banega!
    * Retry-After honor karo - server jo bole utna wait karo
    * Retry budget: total requests me se max 10% hi retries
    * Idempotency keys: retry SAFE ho (payment do baar na lage!)

-----------------------------------------------------------------
SECTION 7: STRATEGY 3 - RATE LIMITING (Har client ki limit)
-----------------------------------------------------------------

PARAGRAPH:
Load shedding bolti hai: "System OVERLOADED hai, excess reject."
Rate limiting bolti hai: "Ye KHAS client zyada bhej raha hai,
usi ko throttle karo." Matlab rate limiting FAIRNESS aur ABUSE
control ke liye hai - koi ek user/bot poori capacity kha na le.
Jaise paani ki tanki se sabko barabar paani mile, koi ek pipe
laga ke sab na nikaal le! 429 Too Many Requests return hota hai.

DIAGRAM:

    Client A (100 req/min - OK)  --under limit-->  [RATE LIMITER] --> Server
    Client B (50 req/min  - OK)  --under limit-->  [RATE LIMITER] --> Server
    Client C (500 req/min - OVER!) --X-->  429 Too Many Requests!

    LIMITS KAHAN-KAHAN LAGAO (defense in depth):

    Request --> [Global limit]  --> 503  (poore cluster ka budget)
            --> [Per-IP limit]  --> 429  (scrapers/bots block)
            --> [Per-User limit]--> 429  (100 req/min per user)
            --> [Per-Endpoint]  --> 429  (search mehenga - 10/min)
            --> [Server] (ab safe hai!)

BULLET POINTS:
   * Rate limiting = fairness + abuse control
   * Load shedding = emergency brake | Rate limiting = traffic rules
   * Algorithms:
       - TOKEN BUCKET (best): tokens dheere-dheere refill hote
         hain, har request 1 token khaati hai. Short BURST allowed
         - real users ke pattern se match karta hai
       - LEAKY BUCKET: constant smooth rate pe drain
       - FIXED WINDOW: simple, par window boundary pe 2x ho sakta hai
       - SLIDING WINDOW: accurate, par mehenga (memory/compute)
   * Helpful headers return karo (limit kya hai, kitna bacha hai)
   * Khayal raho: rate limiter khud BOTTLENECK na ban jaye!
   * Kamzori: jab LAKHON REAL users ek saath aayein (flash crowd),
     toh per-user limiting zyada help NAHI karti

-----------------------------------------------------------------
SECTION 8: STRATEGY 4 - CACHING (Read spikes ka superhero)
-----------------------------------------------------------------

PARAGRAPH:
Read-heavy spike (sab sirf DEKH rahe hain) ke liye caching sabse
effective defense hai. Baar-baar wahi data DB se kyun laayein?
Cache me rakho, DB thanda rakho! Caching LAYERS me kaam karti hai
- browser pe hit hua toh CDN tak nahi jaata, CDN pe hit hua toh
server tak nahi jaata. Do DANGER bhi hain: (1) PREDICTABLE spike
ke liye cache ko PEHLE SE WARM karo (pre-fill), warna cold cache
= DB pe dhamaka. (2) CACHE STAMPEDE - jab popular entry expire
hoti hai toh HAZARON requests ek saath miss karke DB pe gir padti
hain - thundering herd!

DIAGRAM:

    CACHE HIERARCHY (Miss hote jao, aage jao):

    USER --> [Browser Cache] --MISS--> [CDN Edge] --MISS-->
             [App Cache] --MISS--> [Redis] --MISS--> [DATABASE]
                                                          (source of truth)
    Jahan HIT mila - wahin se answer, aage NAHI jaate!

    CACHE STAMPEDE (khatarnaak moment):

    Popular entry EXPIRE hui!
         |
         +--- Request 1 --> MISS --> DB |
         +--- Request 2 --> MISS --> DB |----> DB OVERLOADED!
         +--- Request N --> MISS --> DB |     (Sab ek saath!)

    SOLUTIONS:
    * Staggered TTL: sab entries 300 sec pe expire na ho -
      random(300 +/- 30) - dheere-dheere expire hon
    * Stale-while-revalidate: purana (thoda stale) data turant
      do, background me ek request refresh karti rahe
    * Request coalescing / distributed lock: MISS pe sirf EK
      request DB jaye, baaki uska wait karein ya stale de dein

BULLET POINTS:
   * Cache layers: Browser -> CDN -> App cache -> Redis -> DB
   * Har layer apne peeche wali layer ka LOAD kam karti hai
   * PREDICTABLE spike? Cache WARM karo pehle:
       1. Sale ke products identify karo
       2. Redis me pre-populate karo
       3. CDN ko prime karo (har product page ek baar request karo)
       4. Hit rate verify karo
       5. TTL spike ke poore time tak chalne layak rakho
   * Caching write-heavy problems SOLVE nahi karti
     (purchase cache nahi hota bhai!)
   * Cons: stale data ka risk, invalidation complex, cold
     cache restart ke baad dangerous

-----------------------------------------------------------------
SECTION 9: STRATEGY 5 - QUEUE-BASED LOAD LEVELING (Write spikes ka ilaaj)
-----------------------------------------------------------------

PARAGRAPH:
Write-heavy spike (sab ORDER kar rahe hain) me cache madad nahi
karti - purchase ko cache nahi kar sakte! Lekin flow ASYNC ho
sakta hai toh QUEUE use karo. Queue "KAAM LENA" aur "KAAM KARNA"
ko alag kar deti hai. API sirf request VALIDATE karke queue me
daal deta hai aur turant "202 Accepted" (pending) return karta
hai. Workers queue se apni speed se kaam karte hain. DB ko raw
spike nahi, CONTROLLED write rate dikhta hai. Jaise ticket counter
pe line - sab token le lete hain (accepted), counter apni speed
se kaam karta hai.

DIAGRAM:

    TRAFFIC SPIKE (10,000 orders!)
         |
         v
    [API SERVER] - validate + dedupe + 202 return (FAST, stateless)
         |
         v  (enqueue - kaam line me daal do)
    +---------------------------+
    |    MESSAGE QUEUE          |  Kafka / SQS / RabbitMQ
    |  [req][req][req][req]...  |  (work BUFFER ho gaya)
    +---------------------------+
         |          |         |
         v          v         v
     [Worker]   [Worker]   [Worker]  <-- apni SUSTAINABLE speed se
         |          |         |          process karte hain
         +----------+---------+
                    |
                    v
               [DATABASE]  <-- raw spike nahi, controlled rate!

    BACKPRESSURE (jab queue khud full ho jaye):
    Queue < 50%   --> normally accept
    Queue 50-80%  --> accept + ops team ko ALERT
    Queue 80-95%  --> accept slow karo (delays add karo)
    Queue > 95%   --> 503 reject + Retry-After

BULLET POINTS:
   * 202 Accepted = "kaam ACCEPT hua" - "purchase COMPLETE hua"
     NAHI! Ye difference samajhna zaroori hai
   * Client status POLL/webhook/order-page se check karta hai
   * Queue types: FIFO (fair), Priority (VIP pehle), Delay,
     Dead-letter (fail hue messages yahan - debugging ke liye)
   * BEST for: order fulfillment (order save ho jane ke baad),
     emails/notifications, reports, bulk imports
   * NOT for: real-time search, instant result chahiye, aur
     SCARCE INVENTORY/payments (jab tak reservation +
     idempotency + dedup na ho)
   * Queue depth + queue AGE monitor karo (purane messages =
     warning!)
   * Pros: spike buffer, workers crash ho toh kaam queue me
     SAFE rehta hai, workers ko alag scale kar sakte ho
   * Cons: async response, infra complexity, idempotency
     handle karna zaroori

-----------------------------------------------------------------
SECTION 10: STRATEGY 6 - GRACEFUL DEGRADATION (Thoda hi sahi, par chalta rahe)
-----------------------------------------------------------------

PARAGRAPH:
Jab load aapki saari defenses paar kar jaye, toh system ko POORA
fail hone se bachao - PARTIALLY fail karne do. Core principle:
"core flows fail hone se pehle reduced experience do." Load
shedding PURA request reject karti hai; degradation request ko
serve karta rakhta hai par andar ki features OFF/simplify kar
deta hai. Jaise flight me emergency ho toh khana-band, par
engine aur oxygen chalte rahenge! Product page load ho, bas
"recommended for you" strip ki jagah khali aa jaye - theek hai!

DIAGRAM:

    DEGRADATION LEVELS (load ke hisab se features OFF):

    Load < 70%  -->  NORMAL: sab features ON
    Load > 70%  -->  ELEVATED: P3 OFF (analytics, A/B tests)
                               (user ko kuch pata nahi chalta!)
    Load > 85%  -->  HIGH: P2 bhi OFF (recommendations,
                           personalization), P1 simplify
                           (static/simplified pages)
    Load > 95%  -->  SURVIVAL: sirf P0 (checkout, payment, auth)
                               BARE MINIMUM experience

    +------------------+     +---------------------+
    | System Metrics   | --> | Degradation         |
    | CPU/Latency/Error|     | Controller          |
    +------------------+     +----------+----------+
                                        | feature flags flip karo
                                        v
                             Features auto ON/OFF hote hain

BULLET POINTS:
   * Feature priority PEHLE decide karo (crisis me nahi!):
       P0: Checkout, Payments, Auth     - protect longest
       P1: Search, Product pages, Cart  - core experience
       P2: Recommendations, Reviews     - nice to have
       P3: Analytics, A/B tests         - sabse pehle band
   * Implementation:
       - FEATURE FLAGS: crisis me flag FLIP karo, code deploy nahi!
       - STATIC FALLBACKS: key pages ke static versions pehle
         se bana lo, CDN se serve karo
       - TIMEOUTS + FALLBACK: service slow? degraded response
         do, caller ko hang mat karo
   * Shedding + Degradation SAATH chalte hain:
     shedding total capacity bachata hai, degradation accepted
     requests ke ANDAR experience bachata hai
   * Pros: partial service > complete failure, load kam hote
     hi auto-recover
   * Cons: bahut saare code paths test karne padte hain

-----------------------------------------------------------------
SECTION 11: STRATEGY 7 - DATABASE PROTECTION (Sabse kamzor kadi)
-----------------------------------------------------------------

PARAGRAPH:
Spike me DB sabse pehle akadta hai, kyunki stateless app servers
tez scale ho jate hain par DB nahi! 4+ hathiyar hain:
(1) CONNECTION POOLING - har connection DB ke liye mehenga hai
(ek memory + process khata hai). 100 servers x 20 connections =
2000 connections = DB mar jayega! PgBouncer jaisa pooler beech
me baithkar 2000 client connections ko sirf ~100 REAL connections
me multiplex kar deta hai. (2) READ REPLICAS - writes PRIMARY
ko, reads REPLICAS ko. Par dhyan: replication LAG hota hai (replica
purana data dikha sakta hai). (3) CIRCUIT BREAKER - DB fail ho
raha hai toh aur requests mat bhejo! (4) QUERY TIMEOUTS - kabhi
bhi infinite wait mat karo. Sustained WRITE growth ke liye
SHARDING hi asli jawab hai.

DIAGRAM:

    (1) CONNECTION POOLING:
    Server 1..100 (har ek 20 conns mangta hai = 2000!)
         |
         v
    [PgBouncer / ProxySQL]  <-- multiplexer ( adaptor )
         |
         v
    Database: sirf ~100 REAL connections  <-- DB SAFE!

    (2) READ REPLICAS:
    Application --> Query Router --+
         |                         |
    writes |                  reads |
         v                         v
    [PRIMARY] --replication--> [Replica 1] [Replica 2] [Replica 3]
    (INSERT/UPDATE/DELETE)      (SELECT queries yahan!)

    (3) CIRCUIT BREAKER (bijli ka breaker jaisa):

    CLOSED ----(5 failures/10s)----> OPEN ----(30s cooldown)---->
    (normal,                      (saare DB calls reject,
     sab pass)                     cached/fallback do)
                                       |
                                       v
                                  HALF-OPEN --> ek test request
                                       |             |
                                  success --> CLOSED (wapas normal)
                                  failure --> OPEN (aur cooldown)

BULLET POINTS:
   * Connection pooling = connections SASTI banani, DB ki
     memory bachana (PgBouncer, ProxySQL)
   * Read replicas = READ capacity badhate hain, WRITE nahi!
     Fresh data chahiye toh PRIMARY se padho (lag issue)
     New replica ko warm hone me ghante lag sakte hain -
     PEHLE SE add karo, incident ke beech me nahi
   * Circuit breaker = failing dependency ko TODO band karo,
     use RECOVER karne ka time do
   * BULKHEAD: har dependency ka ALAG thread pool - payment slow
     ho toh search/checkout na doobe (ship ke compartments jaisa)
   * Query timeout (jaise 5 sec) -> timeout pe cache/fallback ->
     KABHI infinite wait nahi
   * WRITE ceiling badhani hai toh:
       - SHARDING: data ko multiple primaries me baanto (asli
         write capacity X hoti hai) - SHARD KEY dhyan se! Hot
         product ID pe shard kiya toh bottleneck wapas aa gaya
       - Write batching: kai rows ek saath insert
       - Async fan-out: ek durable write + baad me counters/
         indexes derive karo
       - Write-optimized DBs: Cassandra, ScyllaDB (LSM-tree)

-----------------------------------------------------------------
SECTION 12: STRATEGY 8 - PRE-WARMING & CAPACITY PLANNING
-----------------------------------------------------------------

PARAGRAPH:
Predictable spike ke liye TAIYARI > clever incident response.
Sale kab shuru hogi aapko pata hai - toh pehle se scale karo!
Capacity MATH samjho: expected peak 100k req/sec, current capacity
20k, safety buffer 1.5x lagao = 150k chahiye = current ka 7.5x.
Toh app servers 10 -> 75, read replicas 2 -> 8, cache 16GB -> 80GB.
Ek line me interview answer: "PRE-SCALE, PRE-WARM caches+CDN,
load test at peak, runbook ready."

DIAGRAM:

    PREPARATION TIMELINE:

    Day -7        Day -3          Day -1        Hour -2     EVENT
      |             |               |              |          |
      v             v               v              v          v
    [Capacity    [Infra          [Caches       [War room   [MONITOR
     planning]    scaled-up]      WARMED +      + team      & respond,
     7.5x calc    servers,        load test]    ready,      flags toggle
     karke        replicas,                     runbook     karke]
     resource     cache                         review]
     list]        badhao]

BULLET POINTS:
   * Capacity = expected peak x 1.5 safety buffer
   * Auto-scaling pe RELY MAT karo predictable events me -
     SCHEDULED scaling karo (pehle se online + warmed!)
   * Day-7: planning | Day-3: infra scale | Day-1: warming +
     load test | Hour-2: war room | Event: monitor
   * Scale-DOWN bhi DHEERE-DHEERE karo (event +4 hrs se start,
     +24 hrs me normal)
   * Feature flags ready rakho toggle karne ke liye
   * Post-mortem ke liye issues DOCUMENT karte jao

-----------------------------------------------------------------
SECTION 13: MASTER DIAGRAM (Sab strategies ek saath)
-----------------------------------------------------------------

    USERS (spike! 10,000 req/sec)
      |
      v
    [RATE LIMITER] ----------> abusers/ko throttle (429)
      |
      v
    [LOAD SHEDDER] ----------> overload pe P3/P2 reject (503)
      |
      v
    [CDN / CACHES] ----------> read-heavy traffic YAHIN katam!
      |                         (pre-warmed, staggered TTL)
      v
    [AUTO-SCALING SERVERS] ---> lag ke dauran bhi zinda rahna
      |                         (scaling gap ki defense upar hai)
      v
    [QUEUE] ------------------> write-heavy work async buffer
      |                         (202 return, backpressure)
      v
    [WORKERS] --> [PgBouncer] --> [PRIMARY DB]
      |                              |
      |                        [READ REPLICAS]
      |
    [CIRCUIT BREAKER + TIMEOUTS + BULKHEADS] sab jagah lagaye hue!
      |
    [FEATURE FLAGS] -> zaroorat pe features OFF (graceful
                       degradation)

-----------------------------------------------------------------
QUICK REVISION (Interview one-liners - yaad kar lo!)
-----------------------------------------------------------------
   * Spike = demand capacity se tez badhna; har spike ka
     alag ilaaj (predictable/unpredictable/self-inflicted)
   * Overload me latency NON-LINEAR badhti hai + failures
     CASCADE karte hain (DB slow -> threads block -> retry storm)
   * Auto-scaling zaroori par KAAFI NAHI - scaling lag me
     3-5 min lagte hain, us gap me bachna padta hai
   * Load shedding = sabki buri service se achhi hai "important
     ko achha, baaki ko jaldi 503 + Retry-After"
   * Priorities: P0 payments/auth LAST shed, P3 analytics PEHLE
   * Rate limiting = FAIRNESS (per-user/IP/endpoint), load
     shedding = EMERGENCY BRAKE
   * Token bucket = bursts allow karta hai (real users ke liye best)
   * Client side bhi responsible hai: backoff + JITTER +
     idempotency (warna retry storm!)
   * Caching = read spikes ka superhero; lekin CACHE STAMPEDE
     se bacho (staggered TTL, stale-while-revalidate, locking)
   * Predictable spike? Cache + CDN ko PEHLE SE WARM karo
   * Queue = "kaam lena" aur "kaam karna" alag kar do (writes
     ke liye); 202 = accepted, COMPLETED nahi!
   * Queue full = BACKPRESSURE (accept slow/reject karo)
   * Graceful degradation = core flows fail hone se PEHLE
     features band karo (feature flags!)
   * DB protection: pooling (PgBouncer), read replicas (lag
     ka dhyan), circuit breaker, timeouts, sharding (writes ke
     liye - shard key SMART choose karo!)
   * Predictable spike = PRE-SCALE, PRE-WARM, load test,
     runbook ready - taiyari is the best defense!
=================================================================
*/